# Lab - Structured Output: Parse It Defensively

Three tasks on reading Claude's structured output safely: a schema that fixes the shape, a parser that refuses a reply that did not finish, and a check for values that fit the schema but mean nothing.

| Task | What you learn |
| --- | --- |
| 1 | A JSON schema, and a typed record on the application's side |
| 2 | read_json(): check stop_reason before json.loads, fail with a reason |
| 3 | Placeholder values like "unknown" are well formed - and wrong |

**You do not need an API key.** This lab ships with a Claude simulator, so every cell
runs offline. The code you write is exactly the code you would write against the real
API - set `ANTHROPIC_API_KEY` at home and the same notebook calls Claude for real.

**You do not write code from scratch.** Each cell already holds the code, with blanks
marked `...` and a comment telling you what goes in each one.

## Setup

Run this cell first, before anything else. Click it, then press **Shift+Enter**.

In [ ]:
# --- Lab setup (provided - just run it) ---
import shopassist_lab
from shopassist_lab import check

# Everything below this line is ordinary Claude API code.
from dotenv import load_dotenv
from anthropic import Anthropic
import json
import re
from dataclasses import dataclass

load_dotenv()
client = Anthropic()

# The model name lives in one constant, so a later update changes one line.
MODEL = "claude-sonnet-5-5"
INSTRUCTION = "Extract the return request from this customer email."

EMAILS = [
    "Hi, I want to return the blue jacket from order "
    "ORD-12345678. It arrived last week but it is too small. Alex",
    "Hello, the shoes I got are damaged. Please tell me how "
    "to send them back. Thanks, Sam",
]

---

## Task 1 - A schema, and a typed record in code

In [ ]:
# ============================================================
# TASK 1 - A schema, and a typed record in code
# ============================================================
#
# WHAT TO DO
#   Finish the JSON schema of a return request, and the record
#   type the application keeps it in. Extract the first email
#   and turn the parsed JSON into a ReturnRequest.
#
# WHY IT MATTERS
#   A support system does not want prose - it wants fields it
#   can store. The schema describes the shape: three required
#   fields, a reason limited to four values, nothing else. On
#   your side, the same shape is a typed record: the rest of the
#   code works with request.order_id, not with a dict that might
#   have any keys.
#
# WHERE TO SEE IT IN THE LECTURE
#   "The shape is a JSON schema".
#
# HOW TO DO IT
#   Three blanks:
#
#       ["too_small", "too_big", "damaged", "other"]   the allowed reasons
#       False   no fields beyond the three
#       data    the parsed dict, unpacked into the record
# ============================================================

@dataclass
class ReturnRequest:   # provided: the shape, on the application's side
    order_id: str
    item: str
    reason: str


def ask(email, max_tokens=300):   # provided: the request with the output format
    return client.messages.create(
        model=MODEL,
        max_tokens=max_tokens,
        output_config={"effort": "low",
                       "format": {"type": "json_schema", "schema": SCHEMA}},
        messages=[{"role": "user", "content": INSTRUCTION + "\n\n" + email}],
    )


try:
    # TODO: replace each ... below with the value named beside it
    SCHEMA = {
        "type": "object",
        "properties": {
            "order_id": {"type": "string"},
            "item": {"type": "string"},
            "reason": {"type": "string",
                       "enum": ...},   # ["too_small", "too_big", "damaged", "other"]
        },
        "required": ["order_id", "item", "reason"],
        "additionalProperties": ...,   # False
    }
    good = ask(EMAILS[0])
    data = json.loads(good.content[0].text)
    first = ReturnRequest(**...)   # data
except (TypeError,) as exc:   # provided: a blank left above, said plainly
    if "llipsis" in str(exc):
        raise ValueError("You still have ... above. Replace each one with the value "
                         "named beside it.") from None
    raise
print(first)

check("schema_record", schema=SCHEMA, first=first)

---

## Task 2 - Read stop_reason before json.loads

In [ ]:
# ============================================================
# TASK 2 - Read stop_reason before json.loads
# ============================================================
#
# WHAT TO DO
#   Finish read_json(): it refuses a reply that did not finish
#   - with UnusableReply, naming the stop reason - and only then
#   parses the text block. Try it on a reply cut short with
#   max_tokens=10.
#
# WHY IT MATTERS
#   With a JSON schema in the output format, the reply is JSON
#   that matches it - when Claude finishes. A reply cut by
#   max_tokens is half an object, and json.loads fails on it
#   with an error that says nothing about why. Read stop_reason
#   first, and fail with a reason your logs and your retry logic
#   can act on.
#
# WHERE TO SEE IT IN THE LECTURE
#   "So we take the text block and parse it with json loads".
#
# HOW TO DO IT
#   Three blanks:
#
#       "end_turn"        the only stop reason that means a whole object
#       "text"            the type of the block that holds the JSON
#       json.loads        parse the text, now that it is complete
# ============================================================


class UnusableReply(Exception):   # provided
    """The reply cannot be parsed into a record."""


# TODO: replace each ... below with the value named beside it
def read_json(response):
    if response.stop_reason != ...:   # "end_turn"
        raise UnusableReply(f"stop_reason is {response.stop_reason}")
    for block in response.content:
        if block.type == ...:   # "text"
            return json.loads(block.text)
    raise UnusableReply("no text block in the reply")

cut = ask(EMAILS[0], max_tokens=10)   # provided: a ceiling the JSON cannot fit under
print("the cut reply:", repr(cut.content[0].text), "| stop_reason:", cut.stop_reason)
try:
    read_json(cut)
except UnusableReply as exc:
    print("refused:", exc)
except json.JSONDecodeError as exc:   # provided: what happens without the check
    print("json.loads failed with no reason given:", exc)

check("parse_defensively", read_json=read_json, cut=cut, error_type=UnusableReply, good=good)

---

## Task 3 - Well formed is not correct

In [ ]:
# ============================================================
# TASK 3 - Well formed is not correct
# ============================================================
#
# WHAT TO DO
#   Extract both emails, then finish problems_of(): it lists the
#   fields whose value is a placeholder - "unknown", "n/a", an
#   empty string - or an order ID in the wrong format.
#
# WHY IT MATTERS
#   The second email never mentions an order number, so its
#   order_id comes back as the word "unknown". The schema was
#   satisfied: a string was required, and a string came back.
#   Stored, it would be a return request that points at no
#   order. A schema makes the answer well formed; correct values
#   are a separate problem, and the application must check them.
#
# WHERE TO SEE IT IN THE LECTURE
#   "It is tempting to think a schema makes the answer correct".
#
# HOW TO DO IT
#   Three blanks:
#
#       PLACEHOLDERS     the values that mean "no value"
#       r"ORD-\d{8}"     what a real order ID looks like
#       read_json(ask(email))   each email, parsed defensively
# ============================================================

PLACEHOLDERS = {"", "unknown", "n/a", "none", "not provided"}   # provided

try:
    # TODO: replace each ... below with the value named beside it
    def problems_of(data):
        bad = [name for name, value in data.items()
               if str(value).strip().lower() in ...]   # PLACEHOLDERS
        if "order_id" not in bad and not re.fullmatch(..., data["order_id"]):   # r"ORD-\d{8}"
            bad.append("order_id")
        return bad
    
    records = [... for email in EMAILS]   # read_json(ask(email))
    problems = [problems_of(data) for data in records]
except (TypeError, AttributeError) as exc:   # provided: a blank left above, said plainly
    if "llipsis" in str(exc):
        raise ValueError("You still have ... above. Replace each one with the value "
                         "named beside it.") from None
    raise
for data, found in zip(records, problems):
    print(data, "->", "needs review: " + ", ".join(found) if found else "ok")

check("sentinel_values", problems=problems)

---

## Done

Use a schema to make the output parse every time, read stop_reason before you parse, and still validate the values before you trust them - a well-formed placeholder is still a missing value.